# BPIC17 (Camargo) — Old vs Improved

Single-source comparison: runs MC suffix sampling for both checkpoints (skipping any side that
already has chunked outputs), evaluates the same metric set Henryk uses in
`evaluation_metric_notebooks/improved/`, then plots them side by side.

**Pair:** `camargo/bpic17` &nbsp;·&nbsp; **Activity key:** `concept:name` &nbsp;·&nbsp; **Samples/case:** `1000`

> **Notes:** Camargo uses a different model class than DropoutUncertaintyEncoderDecoderLSTM. Replace `ensure_sampled` with a Camargo-specific loader before running.

## How to run

1. Confirm the paths in the parameter cell point at the right checkpoints + test pickles.
2. Execute top-to-bottom. Sampling is skipped automatically if `SAMPLED_DIR_*` already contains `results_part_*.pkl`.
3. The bottom cells (table + overlay plot) tolerate either side being `None` — useful while the improved
   checkpoint is still being trained.


## Parameters

In [ ]:
# === PARAMETER CELL ===
# Tweak paths and knobs here. Everything below should run unchanged.
from pathlib import Path

PAIR_DIR = Path('.').resolve()   # the dir containing this notebook

# --- model checkpoints (None = side not yet trained / placeholder) ---
MODEL_OLD_PATH      = PAIR_DIR / 'old/Training/pkl/BPIC17_camargo_ngram15.pkl'
MODEL_IMPROVED_PATH = None

# --- encoded test pickles ---
TEST_PKL_OLD      = PAIR_DIR / 'old/Loader/pkl/BPIC_2017_all_5_test.pkl'
TEST_PKL_IMPROVED = None

# --- where the chunked sampling results land (also where batch_evaluate reads from) ---
SAMPLED_DIR_OLD      = PAIR_DIR / 'evaluation_results/old'
SAMPLED_DIR_IMPROVED = PAIR_DIR / 'evaluation_results/improved'

# --- comparison output ---
COMPARISON_PKL = PAIR_DIR / 'bpic17_old_vs_improved.pkl'
CAPTION        = 'BPIC17 (Camargo)'

# --- sampling knobs (ProbabilisticEvaluation kwargs) ---
CONCEPT_NAME        = 'concept:name'
ALL_CAT             = ['concept:name', 'org:resource', 'lifecycle:transition']
ALL_NUM             = ['case_elapsed_time', 'event_elapsed_time']
GROWING_NUM_VALUES  = ['case_elapsed_time']
NUM_PROCESSES       = 32
SAMPLES_PER_CASE    = 1000
SAVE_EVERY          = 500
RANDOM_ORDER        = False
USE_VARIANCE_CAT    = True
USE_VARIANCE_NUM    = True
SAMPLE_ARGMAX       = False

# --- metric knobs ---
ACTIVITY_KEY      = 'concept:name'
EVENT_LABEL_LIST  = ['A_Accepted', 'A_Cancelled', 'A_Complete', 'A_Concept', 'A_Create Application', 'A_Denied', 'A_Incomplete', 'A_Pending', 'A_Submitted', 'A_Validating', 'O_Accepted', 'O_Cancelled', 'O_Create Offer', 'O_Created', 'O_Refused', 'O_Returned', 'O_Sent (mail and online)', 'O_Sent (online only)', 'W_Assess potential fraud', 'W_Call after offers', 'W_Call incomplete files', 'W_Complete application', 'W_Handle leads', 'W_Personal Loan collection', 'W_Shortened completion ', 'W_Validate application']
VALUE_FACTOR_TIME = 86400   # 3600*24 reports remaining-time in days


## Setup

In [ ]:
import sys, importlib
from pathlib import Path

# Reach `src/` so `model.*`, `src.evaluation_metrics.*` resolve.
_REPO_ROOT = Path('.').resolve()
while not (_REPO_ROOT / 'src').is_dir() and _REPO_ROOT != _REPO_ROOT.parent:
    _REPO_ROOT = _REPO_ROOT.parent
for p in (str(_REPO_ROOT), str(_REPO_ROOT / 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

# Reach the _shared helper module.
_SHARED = _REPO_ROOT / 'src' / 'interpretability' / 'improved_pipeline' / '_shared'
if str(_SHARED) not in sys.path:
    sys.path.insert(0, str(_SHARED))

import comparison_helpers
importlib.reload(comparison_helpers)
from comparison_helpers import (
    SamplingConfig, ensure_sampled, default_metric_set, evaluate_dir,
    comparison_table, plot_overlay, save_results,
)


## MC suffix sampling (skipped per side if chunks already exist)

In [ ]:
sampling_cfg = SamplingConfig(
    concept_name=CONCEPT_NAME,
    growing_num_values=GROWING_NUM_VALUES,
    all_cat=ALL_CAT,
    all_num=ALL_NUM,
    num_processes=NUM_PROCESSES,
    samples_per_case=SAMPLES_PER_CASE,
    sample_argmax=SAMPLE_ARGMAX,
    use_variance_cat=USE_VARIANCE_CAT,
    use_variance_num=USE_VARIANCE_NUM,
    random_order=RANDOM_ORDER,
    save_every=SAVE_EVERY,
)

if MODEL_OLD_PATH and TEST_PKL_OLD:
    ensure_sampled(MODEL_OLD_PATH, TEST_PKL_OLD, SAMPLED_DIR_OLD, sampling_cfg)
else:
    print('[skip] OLD side: missing MODEL_OLD_PATH or TEST_PKL_OLD')

if MODEL_IMPROVED_PATH and TEST_PKL_IMPROVED:
    ensure_sampled(MODEL_IMPROVED_PATH, TEST_PKL_IMPROVED, SAMPLED_DIR_IMPROVED, sampling_cfg)
else:
    print('[skip] IMPROVED side: missing MODEL_IMPROVED_PATH or TEST_PKL_IMPROVED')


## Build metric set

In [ ]:
metrics = default_metric_set(
    activity_key=ACTIVITY_KEY,
    event_label_list=EVENT_LABEL_LIST,
    value_factor_time=VALUE_FACTOR_TIME,
)
print(f'metric set has {len(metrics)} entries')


## Evaluate sampled outputs

In [ ]:
res_old, counts_old = (None, None)
res_improved, counts_improved = (None, None)

if SAMPLED_DIR_OLD.is_dir() and any(SAMPLED_DIR_OLD.glob('results_part_*.pkl')):
    res_old, counts_old = evaluate_dir(SAMPLED_DIR_OLD, metrics)
else:
    print('[skip] OLD eval: no chunks under', SAMPLED_DIR_OLD)

if SAMPLED_DIR_IMPROVED.is_dir() and any(SAMPLED_DIR_IMPROVED.glob('results_part_*.pkl')):
    res_improved, counts_improved = evaluate_dir(SAMPLED_DIR_IMPROVED, metrics)
else:
    print('[skip] IMPROVED eval: no chunks under', SAMPLED_DIR_IMPROVED)


## Side-by-side metric table

In [ ]:
import pandas as pd
df = comparison_table(res_old, res_improved)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
df


## Overlay plots

In [ ]:
plot_overlay(res_old, res_improved, counts_old, counts_improved, caption=CAPTION, pgf=False)


## Save comparison pickle

In [ ]:
save_results(
    COMPARISON_PKL,
    res_old=res_old, counts_old=counts_old,
    res_improved=res_improved, counts_improved=counts_improved,
    config_old=sampling_cfg, config_improved=sampling_cfg,
)


## Notes

- The comparison pickle written in the last cell holds both `(res_raw, counts)` pairs and the sampling configs that produced them. Re-load it later with `comparison_helpers.load_results(path)`.
- To force re-sampling, pass `force=True` into the explicit `ensure_sampled` calls (or just delete the chunked output dir).
- For dataset-specific metric tweaks, copy `default_metric_set` into a cell and edit it; the rest of the pipeline only cares that `metrics` is a `dict[str, metric]`.
